<center>
    <img src="https://rockborne.com/wp-content/uploads/2021/07/LandingPage-Header-RED-CENTRE.jpg" width="900" alt="logo"  />
</center>

# PySpark Hybrid SQL and PySpark Patterns

## PySpark Refresher: Two Ways to Query Data

PySpark gives you two powerful ways to work with data:

1. **DataFrame API** - A programmatic, Python-native approach where you chain methods together
2. **Spark SQL** - Write traditional SQL queries against your data

Both approaches are fully supported, **produce identical performance**, and can be mixed freely. Think of them as two languages that Spark understands equally well, you can speak whichever feels more natural for the task at hand.

**Why does this matter?**
- Teams often have mixed backgrounds (SQL analysts + Python engineers)
- Some operations are clearer in SQL, others in Python
- You'll encounter both in production codebases
- Knowing both makes you more versatile and effective

In [7]:
##This is just for local environments
# Start a SparkSession
import findspark
findspark.init()
findspark.find()

'C:\\spark-3.5.7-bin-hadoop3'

In [8]:
# Cell 1: Setup PySpark Session
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import time
import pandas as pd

# Create Spark session with UI enabled
spark = SparkSession.builder \
    .appName("PySpark Hybrid").getOrCreate()

# Set log level to reduce noise
#spark.sparkContext.setLogLevel("WARN")

print(f"Spark version: {spark.version}")
print(f"Spark UI available at: {spark.sparkContext.uiWebUrl}")

Spark version: 3.5.7
Spark UI available at: http://DESKTOP-V0K0252:4040


In [9]:
#Extracting datasets
import pandas as pd
##Orders dataset: 
pdf = pd.read_csv("https://rockborne-bucket-01-cbs.s3.eu-west-2.amazonaws.com/DataSources/04_02_01_PySpark_Hybrid_Datasets/orders.csv")
orders_df = spark.createDataFrame(pdf)

##Customers dataset:
pdf = pd.read_csv("https://rockborne-bucket-01-cbs.s3.eu-west-2.amazonaws.com/DataSources/04_02_01_PySpark_Hybrid_Datasets/customers.csv")
customers_df = spark.createDataFrame(pdf)

##Products dataset:
pdf = pd.read_csv("https://rockborne-bucket-01-cbs.s3.eu-west-2.amazonaws.com/DataSources/04_02_01_PySpark_Hybrid_Datasets/products.csv")
products_df = spark.createDataFrame(pdf)

In [6]:
del pdf

In [7]:
print(f"Orders: {orders_df.count():,} records")
print(f"Products: {products_df.count():,} records")
print(f"Customers: {customers_df.count():,} records")

# Show sample data
print("\n--- Sample Orders ---")
orders_df.show(5)

print("--- Sample Products ---")
products_df.show(5)

print("--- Sample Customers ---")
customers_df.show(5)

Orders: 500,000 records
Products: 200 records
Customers: 50,000 records

--- Sample Orders ---
+----------+-----------+----------+--------+----------+----------+----------+-------+
|  order_id|customer_id|product_id|quantity|unit_price|order_date|   channel| region|
+----------+-----------+----------+--------+----------+----------+----------+-------+
|ORD0000000|  CUST07297|   PROD007|      12|     86.13|2024-03-22|     Store|  South|
|ORD0000001|  CUST06718|   PROD174|      12|    268.19|2024-04-04|    Online|Central|
|ORD0000002|  CUST02083|   PROD008|       2|      69.5|2024-02-24|    Online|Central|
|ORD0000003|  CUST46926|   PROD167|      12|    165.76|2024-01-26|     Store|   West|
|ORD0000004|  CUST18232|   PROD002|      13|    242.72|2024-03-16|Mobile App|   East|
+----------+-----------+----------+--------+----------+----------+----------+-------+
only showing top 5 rows

--- Sample Products ---
+----------+------------+---------------+----------+
|product_id|product_name|    

## DataFrame API Basics

The DataFrame API lets you manipulate data by chaining method calls. Each method returns a new DataFrame, allowing you to build up complex transformations step by step.

In [ ]:
#DataFrame API - Basic Operations

# SELECTING COLUMNS
# Use .select() to choose which columns to include
#orders_df.select("order_id", "customer_id", "quantity").show(5)
# You can also use col() for more flexibility

#orders_df.select(["order_id", "customer_id", "quantity"]).show(5)
#orders_df.select(F.col("order_id"), F.col("quantity")).show(5)


+----------+-----------+--------+
|  order_id|customer_id|quantity|
+----------+-----------+--------+
|ORD0000000|  CUST07297|      12|
|ORD0000001|  CUST06718|      12|
|ORD0000002|  CUST02083|       2|
|ORD0000003|  CUST46926|      12|
|ORD0000004|  CUST18232|      13|
+----------+-----------+--------+
only showing top 5 rows

+----------+-----------+--------+
|  order_id|customer_id|quantity|
+----------+-----------+--------+
|ORD0000000|  CUST07297|      12|
|ORD0000001|  CUST06718|      12|
|ORD0000002|  CUST02083|       2|
|ORD0000003|  CUST46926|      12|
|ORD0000004|  CUST18232|      13|
+----------+-----------+--------+
only showing top 5 rows



In [12]:
#DataFrame API - Filtering Rows

# FILTERING with .filter() or .where() (they're identical)
# Filter orders with quantity greater than 10
orders_df.filter(F.col("quantity") > 10)
# Multiple conditions - use & (and), | (or), ~ (not)

orders_df.filter(
                (F.col("quantity") > 10) &
                 (F.col("region") == "North")
                  ).count() 
# Always wrap conditions in parentheses!

33772

In [14]:
#DataFrame API - Adding and Modifying Columns
orders_with_total = orders_df.withColumn("line_total", 
                                         F.col("quantity" ) 
                                         * 
                                         F.col("unit_price" ))#.show(5)
# ADDING COLUMNS with .withColumn()
# Calculate line total (quantity * unit_price)

In [16]:
# RENAMING COLUMNS
##Before renaming:
orders_with_total.select(
    F.col("order_id"),
    F.col("quantity").alias("QUANTITY"),
    F.col("unit_price"),
    # Rounding to 2 decimal places and renaming the result
    F.round(F.col("line_total"), 2).alias("rounded_total")
).show(5)

##Creating a new df with the new column:


+----------+--------+----------+-------------+
|  order_id|QUANTITY|unit_price|rounded_total|
+----------+--------+----------+-------------+
|ORD0000000|      12|     86.13|      1033.56|
|ORD0000001|      12|    268.19|      3218.28|
|ORD0000002|       2|      69.5|        139.0|
|ORD0000003|      12|    165.76|      1989.12|
|ORD0000004|      13|    242.72|      3155.36|
+----------+--------+----------+-------------+
only showing top 5 rows



In [ ]:
#DataFrame API - Aggregations
# GROUPING AND AGGREGATING
# Count orders by region

# Multiple aggregations


+-------+------+
| region| count|
+-------+------+
|  South| 99658|
|Central|100135|
|   East| 99856|
|   West| 99709|
|  North|100642|
+-------+------+

+-------+-----------+--------------+------------------+
| region|order_count|total_quantity|         avg_price|
+-------+-----------+--------------+------------------+
|  South|      99658|        795893|152.91201248269059|
|Central|     100135|        801037|152.54697927797483|
|   East|      99856|        798377|152.52618941275443|
|   West|      99709|        797461|152.46144560671559|
|  North|     100642|        806942|152.66081814749282|
+-------+-----------+--------------+------------------+



In [18]:
#DataFrame API - Sorting
# SORTING with .orderBy() or .sort()
# Sort by quantity descending
orders_df.printSchema()
orders_df.columns
# Sort by multiple columns


root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- order_date: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- region: string (nullable = true)



['order_id',
 'customer_id',
 'product_id',
 'quantity',
 'unit_price',
 'order_date',
 'channel',
 'region']

In [24]:
#orders_df.groupBy("region").count().show()
# Multiple aggregations
orders_df.groupBy(["region", "channel"]).agg(
                                    F.count("order_id").alias("order_count"),
                                    F.sum("quantity").alias("total_quantity"),
                                    F.avg("unit_price").alias("avg_price")
).show()

+-------+----------+-----------+--------------+------------------+
| region|   channel|order_count|total_quantity|         avg_price|
+-------+----------+-----------+--------------+------------------+
|   West|     Store|      24922|        199498| 151.9811431666801|
|   West|Mobile App|      24895|        198897|152.75198473589074|
|Central|     Phone|      25224|        201847|152.17607714874728|
|  South|     Phone|      24978|        199875|153.38337216750742|
|Central|    Online|      25316|        203087|152.24906264812753|
|   East|     Phone|      25023|        200659| 153.1062186788155|
|   West|     Phone|      25064|        201305|152.55794845196291|
|  South|     Store|      24769|        197660|152.81332875772134|
|Central|Mobile App|      24936|        198684| 152.8689336701956|
|   East|Mobile App|      24851|        198279| 152.7000354110498|
|  North|Mobile App|      25275|        202563|152.55554856577655|
|  North|     Phone|      24988|        200645|  153.051137345

In [ ]:
orders_df.join(customers_df, on = "customer_id").groupBy(["channel", "customer_name"]).agg(F.sum("quantity")).show(5)


### SQL Basics in PySpark

To use SQL in PySpark, you first register your DataFrame as a "temporary view" - this gives your data a table name that SQL can reference.

In [16]:
#SQL - Registering Views and Basic Queries

orders_df.createOrReplaceTempView("orders")
products_df.createOrReplaceTempView("products")
customers_df.createOrReplaceTempView("customers")
# Register the DataFrame as a temporary view



In [ ]:
# Now you can query it with SQL!
spark.sql("SELECT * " \
            "FROM orders " \
            "LIMIT 5").show()


+----------+-----------+----------+--------+----------+----------+----------+-------+
|  order_id|customer_id|product_id|quantity|unit_price|order_date|   channel| region|
+----------+-----------+----------+--------+----------+----------+----------+-------+
|ORD0000000|  CUST07297|   PROD007|      12|     86.13|2024-03-22|     Store|  South|
|ORD0000001|  CUST06718|   PROD174|      12|    268.19|2024-04-04|    Online|Central|
|ORD0000002|  CUST02083|   PROD008|       2|      69.5|2024-02-24|    Online|Central|
|ORD0000003|  CUST46926|   PROD167|      12|    165.76|2024-01-26|     Store|   West|
|ORD0000004|  CUST18232|   PROD002|      13|    242.72|2024-03-16|Mobile App|   East|
+----------+-----------+----------+--------+----------+----------+----------+-------+



: 

In [6]:
tables_lst = ["orders", "products"]#, "customers"]
for table in tables_lst:
    query = "SELECT * FROM " + table + " LIMIT 5"
    print(query)
    #spark.sql(query).show()

SELECT * FROM orders LIMIT 5
SELECT * FROM products LIMIT 5


In [ ]:
##We start mixing loops and python structures with SQL: 


+----------+-----------+----------+--------+----------+----------+-------+-------+
|  order_id|customer_id|product_id|quantity|unit_price|order_date|channel| region|
+----------+-----------+----------+--------+----------+----------+-------+-------+
|ORD0000000|  CUST07297|   PROD007|      12|     86.13|2024-03-22|  Store|  South|
|ORD0000001|  CUST06718|   PROD174|      12|    268.19|2024-04-04| Online|Central|
|ORD0000002|  CUST02083|   PROD008|       2|      69.5|2024-02-24| Online|Central|
+----------+-----------+----------+--------+----------+----------+-------+-------+
only showing top 3 rows

+----------+------------+-------------+----------+
|product_id|product_name|     category|base_price|
+----------+------------+-------------+----------+
|   PROD001|   Product 1|       Sports|    247.44|
|   PROD002|   Product 2|Home & Garden|    317.44|
|   PROD003|   Product 3|        Books|    218.68|
+----------+------------+-------------+----------+
only showing top 3 rows

+-----------

### Side-by-Side Comparison

Let's see the same query written both ways:


In [ ]:
# Same Query - Two Approaches

# QUESTION: What's the total revenue by region, sorted highest to lowest?
# DataFrame API approach
df_result = (
    orders_df
    .withColumn("line_total", F.col("quantity") * F.col("unit_price"))
    .groupBy("region")
    .agg(F.round(F.sum("line_total"), 2).alias("total_revenue"))
    .orderBy(F.desc("total_revenue"))
)

print("DataFrame API Result:")
df_result.show(5)

# SQL approach
sql_result = spark.sql("""
    SELECT 
        region,
        ROUND(SUM(quantity * unit_price), 2) AS total_revenue
    FROM orders
    GROUP BY region
    ORDER BY total_revenue DESC
""")

print("SQL Result:")
sql_result.show(5)

DataFrame API Result:
+-------+--------------+
| region| total_revenue|
+-------+--------------+
|  North|1.2326413535E8|
|Central|1.2212838627E8|
|  South|1.2171369343E8|
|   East|1.2165980386E8|
|   West| 1.215369955E8|
+-------+--------------+

SQL Result:
+-------+--------------+
| region| total_revenue|
+-------+--------------+
|  North|1.2326413535E8|
|Central|1.2212838627E8|
|  South|1.2171369343E8|
|   East|1.2165980386E8|
|   West| 1.215369955E8|
+-------+--------------+



### Section 1 Exercise: Basic Queries

**Exercise 1.1:** Using BOTH the DataFrame API AND SQL, write queries to answer:

1. How many orders were placed through each channel?
2. What's the average unit_price for orders in the "East" region?
3. Find the top 5 customers by total quantity ordered

In [14]:
# Exercise 1.1 - YOUR SOLUTIONS
# 1a. Descending Orders by channel - DataFrame API
# YOUR CODE HERE
orders_df.groupBy("channel").count().orderBy(F.desc("count"))#.show()

# 1b. Descending Orders by channel - SQL
# YOUR CODE HERE

# 2a. Average unit_price in East - DataFrame API
# YOUR CODE HERE
orders_df.filter(F.col("region") == "East").agg(F.round(F.avg("unit_price"), 2)).show()
# 2b. Average unit_price in East - SQL
# YOUR CODE HERE


# 3a. Top 5 customers by quantity - DataFrame API
# YOUR CODE HERE


# 3b. Top 5 customers by quantity - SQL
# YOUR CODE HERE

+-------------------------+
|round(avg(unit_price), 2)|
+-------------------------+
|                   152.53|
+-------------------------+



## Working with spark.sql() and Temporary Views

### Introduction

When you want to use SQL in PySpark, you need a way to give your DataFrames "names" that SQL can reference. This is where **temporary views** come in. A temporary view is like a named reference to your DataFrame that exists in Spark's catalog, allowing SQL queries to access it by name.

Think of it like this: DataFrames live in Python's memory with variable names. SQL doesn't know about Python variables. Temporary views create a bridge - they register your DataFrame with a name that SQL understands.

### Creating and Managing Temporary Views

PySpark offers two types of temporary views:

| Feature | Local Temporary View | Global Temporary View |
|---------|---------------------|----------------------|
| **Creation** | `createOrReplaceTempView("name")` | `createOrReplaceGlobalTempView("name")` |
| **Scope** | Current SparkSession only | All SparkSessions in the application |
| **Access** | `SELECT * FROM name` | `SELECT * FROM global_temp.name` |
| **Lifetime** | Until session ends or explicitly dropped | Until application ends or explicitly dropped |
| **Use Case** | Most common - single notebook/job | Sharing data between notebooks or sessions |
| **Isolation** | Private to your session | Visible 

In [15]:
# Cell 15: Local Temporary Views

# Creating a local temporary view
orders_df.createOrReplaceTempView("orders")

# Query it directly by name
spark.sql("SELECT COUNT(*) AS total_orders FROM orders").show()

# The "OrReplace" part means if the view already exists, it will be overwritten
# This is useful when you want to update the view with new data

+------------+
|total_orders|
+------------+
|      500000|
+------------+



In [17]:
# You can check what views exist
print("Available tables/views:")
for table in spark.catalog.listTables():
    print(f"  - {table.name} (type: {table.tableType}, isTemporary: {table.isTemporary})")


Available tables/views:
  - customers (type: TEMPORARY, isTemporary: True)
  - orders (type: TEMPORARY, isTemporary: True)
  - products (type: TEMPORARY, isTemporary: True)


In [18]:
# You can also seee the active pyspark dataframes:
from pyspark.sql import DataFrame
dfs_in_memory = [var_name for var_name, obj in globals().items() if isinstance(obj, DataFrame)]
print(f"Found {len(dfs_in_memory)} active DataFrames:")
for df_name in dfs_in_memory:
    print(f"  - {df_name}")

Found 3 active DataFrames:
  - orders_df
  - customers_df
  - products_df


```python
# Cell 16: Global Temporary Views

# Creating a global temporary view
orders_df.createOrReplaceGlobalTempView("orders_global")

# IMPORTANT: Must use global_temp database prefix!
spark.sql("SELECT COUNT(*) AS total FROM global_temp.orders_global").show()

# When would you use global views?
# - When running multiple notebooks that need to share data
# - When different parts of your application need access to the same view
# - When you want the view to persist across SparkSession restarts within the same app

# For most day-to-day work, local views are sufficient
```

In [18]:
# Cell 17: Managing View Lifecycle

# Check if a view exists
print(f"'orders' exists: {spark.catalog.tableExists('orders')}")

'orders' exists: True


In [19]:
# Drop a specific view
spark.catalog.dropTempView("orders")
print(f"After drop - 'orders' exists: {spark.catalog.tableExists('orders')}")

# Drop global view (note: no global_temp prefix needed for drop)
#spark.catalog.dropGlobalTempView("orders_global")


After drop - 'orders' exists: False


In [20]:
# Recreate for later use
orders_df.createOrReplaceTempView("orders")


### Parameterised SQL Queries and SQL Injection

#### What is SQL Injection?

SQL injection is a security vulnerability where an attacker can insert malicious SQL code into a query through user input. This is one of the most common and dangerous web application vulnerabilities.

**Why is this dangerous?**

When you build SQL queries by concatenating strings with user input, a malicious user can "escape" out of your intended query and run their own commands.

In [21]:
# Cell 18: SQL Injection - The Danger Explained

# DANGEROUS EXAMPLE - DO NOT USE IN PRODUCTION!

# Imagine this comes from user input (a web form, API request, etc.)
user_input_category = "Electronics"

# This LOOKS fine...
query = f"SELECT * FROM products WHERE category = '{user_input_category}'"
print(f"Innocent query:\n{query}\n")

# But what if a malicious user enters this?
malicious_input = "Electronics'; DROP TABLE products; --"

# The resulting query becomes:
dangerous_query = f"SELECT * FROM products WHERE category = '{malicious_input}'"
print(f"Malicious query:\n{dangerous_query}\n")

# Breaking down the attack:
# 1. Electronics'     - closes the original string
# 2. ;                - ends the SELECT statement
# 3. DROP TABLE products;  - DELETES YOUR ENTIRE TABLE!
# 4. --               - comments out the rest of the original query

# In a web application, this could:
# - Delete all your data
# - Expose sensitive information
# - Modify records (change prices, grant admin access)
# - Take over your entire database

#print("This is why we NEVER use string formatting with user inputs!")

Innocent query:
SELECT * FROM products WHERE category = 'Electronics'

Malicious query:
SELECT * FROM products WHERE category = 'Electronics'; DROP TABLE products; --'



In [ ]:
# Cell 19: Safe Approaches to Parameterised Queries

# APPROACH 1: Use DataFrame operations (RECOMMENDED)
# Let Spark handle the escaping - completely safe
user_category = "Electronics"

safe_result = products_df.filter(F.col("category") == user_category)
print("Approach 1 - DataFrame filter (safest):")
safe_result.show(3)


Approach 1 - DataFrame filter (safest):
+----------+------------+-----------+----------+
|product_id|product_name|   category|base_price|
+----------+------------+-----------+----------+
|   PROD014|  Product 14|Electronics|    147.75|
|   PROD019|  Product 19|Electronics|    148.06|
|   PROD027|  Product 27|Electronics|     356.6|
+----------+------------+-----------+----------+
only showing top 3 rows



In [ ]:

# APPROACH 2: Validate and sanitise inputs
# Only allow known, expected values
VALID_CATEGORIES = ["Electronics", "Clothing", "Home & Garden", "Sports", "Books", "Food & Beverage"]




Approach 2 - Validated input:
+----------+------------+-----------+----------+
|product_id|product_name|   category|base_price|
+----------+------------+-----------+----------+
|   PROD014|  Product 14|Electronics|    147.75|
|   PROD019|  Product 19|Electronics|    148.06|
|   PROD027|  Product 27|Electronics|     356.6|
+----------+------------+-----------+----------+
only showing top 3 rows



### Multi-Statement SQL Workflows

SQL excels at breaking complex analysis into logical steps using intermediate views or Common Table Expressions (CTEs).

In [19]:
###Views:
#Example Objective: Multi-Stage Data Enrichment
# Goal: Transform raw, fragmented data into a high-level business insight. You will practice "Modular SQL" by breaking a complex problem into three manageable steps:
# Feature Engineering: Calculate a new metric (line_total).
# Data Enrichment: Join disparate datasets (Orders + Customers).
# Aggregation: Extract the final business KPI (Revenue by Membership).

# --- STEP 1: Feature Engineering ---
# We create a 'view' to handle the math so we don't have to repeat it later.
spark.sql("""
    CREATE OR REPLACE TEMPORARY VIEW v_order_calculations AS
    SELECT 
        order_id, 
        customer_id, 
        (quantity * unit_price) AS line_total
    FROM orders
""")

# --- STEP 2: Data Enrichment ---
# We join our calculated view with the customer table to get 'membership_level'.
spark.sql("""
    CREATE OR REPLACE TEMPORARY VIEW v_enriched_orders AS
    SELECT 
        oc.*, 
        c.membership_level
    FROM v_order_calculations oc
    JOIN customers c ON oc.customer_id = c.customer_id
""")

# --- STEP 3: Final Aggregation ---
# Now we query our enriched view to get the final business result.
final_report = spark.sql("""
    SELECT 
        membership_level,
        CAST(SUM(line_total) AS DECIMAL(18, 2)) AS total_revenue,
        COUNT(order_id) AS total_orders
    FROM v_enriched_orders
    GROUP BY membership_level
    ORDER BY total_revenue DESC
""")

# Display the results
print("Final Business Report: Revenue by Membership Level")
final_report.show()

Final Business Report: Revenue by Membership Level
+----------------+-------------+------------+
|membership_level|total_revenue|total_orders|
+----------------+-------------+------------+
|          Bronze| 154703023.44|      126406|
|        Platinum| 152281075.89|      125016|
|          Silver| 151874260.13|      124509|
|            Gold| 151444654.95|      124069|
+----------------+-------------+------------+



In [20]:
# You can check what views exist
print("Available tables/views:")
for table in spark.catalog.listTables():
    print(f"  - {table.name} (type: {table.tableType}, isTemporary: {table.isTemporary})")


Available tables/views:
  - customers (type: TEMPORARY, isTemporary: True)
  - orders (type: TEMPORARY, isTemporary: True)
  - products (type: TEMPORARY, isTemporary: True)
  - v_enriched_orders (type: TEMPORARY, isTemporary: True)
  - v_order_calculations (type: TEMPORARY, isTemporary: True)


In [21]:
#Common Table Expressions (CTEs)
# CTEs let you define temporary result sets within a single query
#Exercise Objective: Advanced SQL Patterns (CTEs)
#Encapsulate Logic: Use the WITH clause to create modular, readable code blocks.
#Multilevel Aggregation: Calculate metrics at the customer level before rolling them up to a regional level.


result = spark.sql("""
    WITH order_totals AS (
        SELECT 
            customer_id,
            region,
            quantity * unit_price AS line_total
        FROM orders
    ),
    customer_summary AS (
        SELECT 
            customer_id,
            region,
            SUM(line_total) AS total_spend,
            COUNT(*) AS order_count
        FROM order_totals
        GROUP BY customer_id, region
    )
    SELECT 
        region,
        COUNT(DISTINCT customer_id) AS unique_customers,
        ROUND(AVG(total_spend), 2) AS avg_customer_spend,
        ROUND(AVG(order_count), 1) AS avg_orders_per_customer
    FROM customer_summary
    GROUP BY region
    ORDER BY avg_customer_spend DESC
""")

print("Customer Analysis by Region (using CTEs):")
result.show()

Customer Analysis by Region (using CTEs):
+-------+----------------+------------------+-----------------------+
| region|unique_customers|avg_customer_spend|avg_orders_per_customer|
+-------+----------------+------------------+-----------------------+
|  North|           43350|           2843.46|                    2.3|
|  South|           43092|           2824.51|                    2.3|
|Central|           43286|           2821.43|                    2.3|
|   East|           43185|           2817.18|                    2.3|
|   West|           43229|           2811.47|                    2.3|
+-------+----------------+------------------+-----------------------+



In [22]:
# You can check what views exist
print("Available tables/views:")
for table in spark.catalog.listTables():
    print(f"  - {table.name} (type: {table.tableType}, isTemporary: {table.isTemporary})")

Available tables/views:
  - customers (type: TEMPORARY, isTemporary: True)
  - orders (type: TEMPORARY, isTemporary: True)
  - products (type: TEMPORARY, isTemporary: True)
  - v_enriched_orders (type: TEMPORARY, isTemporary: True)
  - v_order_calculations (type: TEMPORARY, isTemporary: True)


### Section 2 Exercise: Views and SQL

**Exercise 2.1:** 

1. Create a temporary view called `high_value_orders` that contains only orders where `quantity * unit_price > 500`
2. Query this view to find the top 3 channels by number of high-value orders
3. Create a CTE-based query that finds the average order value by day of week

In [ ]:
#1. Identify Premium Transactions
#Requirement: Create a temporary view named high_value_orders.
#Filter: Include only records where the transaction value (Quantity × Unit Price) exceeds 500.
#Transformation: Add a new column named order_value to represent this calculation so it can be reused in future steps.
# YOUR CODE HERE

In [ ]:
# 1. Create high_value_orders view


1. High value orders view created
   Records: 343,604
+----------+-----------+----------+--------+----------+----------+----------+-------+------------------+
|  order_id|customer_id|product_id|quantity|unit_price|order_date|   channel| region|       order_value|
+----------+-----------+----------+--------+----------+----------+----------+-------+------------------+
|ORD0000000|  CUST07297|   PROD007|      12|     86.13|2024-03-22|     Store|  South|           1033.56|
|ORD0000001|  CUST06718|   PROD174|      12|    268.19|2024-04-04|    Online|Central|3218.2799999999997|
|ORD0000003|  CUST46926|   PROD167|      12|    165.76|2024-01-26|     Store|   West|           1989.12|
|ORD0000004|  CUST18232|   PROD002|      13|    242.72|2024-03-16|Mobile App|   East|           3155.36|
|ORD0000005|  CUST10190|   PROD056|      13|     104.3|2024-02-05|    Online|   West|1355.8999999999999|
+----------+-----------+----------+--------+----------+----------+----------+-------+------------------+



In [ ]:
#2. Market Channel Performance
#Requirement: Analyze the high_value_orders view created in the previous step.
#Metrics: For each sales channel, find the total number of high-value orders and the average order value (rounded to 2 decimals).
#Ranking: Display only the Top 3 channels based on the volume of these premium orders.

In [ ]:
# 2. Query for top 3 channels



2. Top 3 channels by high-value order count:
+----------+----------------+---------------+
|   channel|high_value_count|avg_order_value|
+----------+----------------+---------------+
|     Phone|           86310|        1668.97|
|     Store|           85780|        1665.68|
|Mobile App|           85770|         1665.9|
+----------+----------------+---------------+



In [ ]:
#3. Temporal Analysis (Day of Week)
#Requirement: Using a CTE (Common Table Expression), determine if certain days of the week generate higher-value shopping trips.
#Step A (CTE): Extract the day number and the day name (e.g., 'Monday') from the order_date.
#Step B (Final Query): Group the data by day. Show the total count of orders and the average order value.
#Ordering: The final table must be sorted chronologically (from Sunday to Saturday).

In [ ]:
# 3. CTE for average by day of week


3. Average order value by day of week:
+---------+-----------+---------------+
| day_name|order_count|avg_order_value|
+---------+-----------+---------------+
|   Sunday|      70257|        1221.18|
|   Monday|      74131|        1220.15|
|  Tuesday|      74377|        1223.56|
|Wednesday|      70227|        1224.78|
| Thursday|      70573|         1213.8|
|   Friday|      70169|        1224.63|
| Saturday|      70266|        1216.04|
+---------+-----------+---------------+



## 3. DataFrame API Strengths

### Introduction

While SQL is powerful and familiar to many, the DataFrame API offers unique advantages that make it the preferred choice in certain scenarios. The key strengths are:

1. **Programmatic control** - Build queries dynamically based on conditions
2. **Reusable functions** - Create transformation libraries you can test and reuse
3. **IDE support** - Get autocomplete, type checking, and better debugging
4. **Integration** - Works seamlessly with Python libraries and ML pipelines

This section explores when and how to leverage these strengths.

### Programmatic Control and Composition

#### Dynamic Column Selection

Sometimes you don't know at coding time which columns you need - maybe a user selects them, or a config file specifies them.

In [24]:
# Columns the user selected (imagine this comes from a UI)
selected_columns = ["order_id", "customer_id"]
orders_df.select(selected_columns).show()


+----------+-----------+
|  order_id|customer_id|
+----------+-----------+
|ORD0000000|  CUST07297|
|ORD0000001|  CUST06718|
|ORD0000002|  CUST02083|
|ORD0000003|  CUST46926|
|ORD0000004|  CUST18232|
|ORD0000005|  CUST10190|
|ORD0000006|  CUST23527|
|ORD0000007|  CUST08181|
|ORD0000008|  CUST46175|
|ORD0000009|  CUST15257|
|ORD0000010|  CUST23284|
|ORD0000011|  CUST11216|
|ORD0000012|  CUST41944|
|ORD0000013|  CUST02104|
|ORD0000014|  CUST47050|
|ORD0000015|  CUST17360|
|ORD0000016|  CUST38243|
|ORD0000017|  CUST05958|
|ORD0000018|  CUST44597|
|ORD0000019|  CUST34677|
+----------+-----------+
only showing top 20 rows



In [25]:
# Add computed columns dynamically
selected_columns = ["order_id", "customer_id", "region"]
if "quantity" in orders_df.columns and "unit_price" in orders_df.columns:
    selected_columns.append("quantity")
    selected_columns.append("unit_price")

orders_df.select(selected_columns).show()



+----------+-----------+-------+--------+----------+
|  order_id|customer_id| region|quantity|unit_price|
+----------+-----------+-------+--------+----------+
|ORD0000000|  CUST07297|  South|      12|     86.13|
|ORD0000001|  CUST06718|Central|      12|    268.19|
|ORD0000002|  CUST02083|Central|       2|      69.5|
|ORD0000003|  CUST46926|   West|      12|    165.76|
|ORD0000004|  CUST18232|   East|      13|    242.72|
|ORD0000005|  CUST10190|   West|      13|     104.3|
|ORD0000006|  CUST23527|   West|      10|     83.03|
|ORD0000007|  CUST08181|Central|       2|    167.85|
|ORD0000008|  CUST46175|  North|       1|    200.07|
|ORD0000009|  CUST15257|  South|       7|      87.0|
|ORD0000010|  CUST23284|Central|      11|     83.76|
|ORD0000011|  CUST11216|   West|      12|     77.22|
|ORD0000012|  CUST41944|  North|       9|     69.79|
|ORD0000013|  CUST02104|Central|       7|     83.98|
|ORD0000014|  CUST47050|   West|       4|    198.35|
|ORD0000015|  CUST17360|Central|       4|    2

In [ ]:
def dynamic_col_sel(df, selected_columns):
    if "quantity" in df.columns and "unit_price" in df.columns:
        selected_columns.append("quantity")
        selected_columns.append("unit_price")
        return df.select(selected_columns)
    else:
        return df.select(selected_columns)
    #orders_df.select(selected_columns).show()
    

In [ ]:
##Finally we can make a function of this: 
orders_df.columns

['order_id',
 'customer_id',
 'product_id',
 'quantity',
 'unit_price',
 'order_date',
 'channel',
 'region']

In [29]:
selected_columns = ["order_id", "customer_id", "region"]
dynamic_col_sel(orders_df, selected_columns).show(3)

+----------+-----------+-------+--------+----------+
|  order_id|customer_id| region|quantity|unit_price|
+----------+-----------+-------+--------+----------+
|ORD0000000|  CUST07297|  South|      12|     86.13|
|ORD0000001|  CUST06718|Central|      12|    268.19|
|ORD0000002|  CUST02083|Central|       2|      69.5|
+----------+-----------+-------+--------+----------+
only showing top 3 rows



In [30]:
##Lest see anothe tabke that doesnt have quantity or unit price:
customers_df.columns

['customer_id', 'customer_name', 'membership_level', 'join_year']

In [32]:
selected_columns = ["customer_name", "membership_level"]
dynamic_col_sel(customers_df, selected_columns).show(3)

+-------------+----------------+
|customer_name|membership_level|
+-------------+----------------+
|   Customer 1|        Platinum|
|   Customer 2|          Bronze|
|   Customer 3|            Gold|
+-------------+----------------+
only showing top 3 rows



In [34]:
# Dynamic Filtering

# Scenario: Build filters based on user input

# User-selected filters (from a dashboard, API, etc.)
filters = {
    "region": "North",
    "channel": "Online"
}

# Start with full dataset
result = orders_df

# Apply each filter dynamically
for column, value in filters.items():
    result = result.filter(F.col(column) == value)

print(f"After applying filters: {result.count():,} records")
result.count()

After applying filters: 25,014 records


25014

In [ ]:
##We can make it a function:


In [55]:
dynamic_filtering(orders_df).show(5)

Before applying filters: 500,000 records
After applying filters: 25,014 records
+----------+-----------+----------+--------+----------+----------+-------+------+
|  order_id|customer_id|product_id|quantity|unit_price|order_date|channel|region|
+----------+-----------+----------+--------+----------+----------+-------+------+
|ORD0000019|  CUST34677|   PROD065|       9|    258.93|2024-02-29| Online| North|
|ORD0000030|  CUST38556|   PROD142|       4|    178.59|2024-01-03| Online| North|
|ORD0000035|  CUST47781|   PROD014|      11|    197.77|2024-02-29| Online| North|
|ORD0000038|  CUST04941|   PROD114|      13|     259.1|2024-02-01| Online| North|
|ORD0000053|  CUST43476|   PROD027|      15|     44.62|2024-02-08| Online| North|
+----------+-----------+----------+--------+----------+----------+-------+------+
only showing top 5 rows



#### Example: Dynamic Aggregations


In [ ]:
# Dynamic Aggregation Builder
# Scenario: A reporting tool where users select which metrics to include
# Available metrics and their definitions
METRIC_DEFINITIONS = {
    "total_orders": F.count("order_id"),
    "total_revenue": F.sum(F.col("quantity") * F.col("unit_price")),
    "avg_order_value": F.avg(F.col("quantity") * F.col("unit_price")),
    "max_order_value": F.max(F.col("quantity") * F.col("unit_price")),
    "unique_customers": F.countDistinct("customer_id"),
    "avg_quantity": F.avg("quantity")
}

def build_report(df, group_by_cols, metric_names):
    """
    Build a dynamic report based on user selections.
    
    Args:
        df: Source DataFrame
        group_by_cols: List of columns to group by
        metric_names: List of metric names to include
    
    Returns:
        Aggregated DataFrame
    """
    # Build list of aggregation expressions
    agg_expressions = []
    for name in metric_names:
        if name in METRIC_DEFINITIONS:
            agg_expressions.append(
                F.round(METRIC_DEFINITIONS[name], 2).alias(name)
            )
    
    # Apply grouping and aggregation
    return df.groupBy(group_by_cols).agg(*agg_expressions)

In [ ]:
# User request 1: Revenue by region
print("Report 1: Revenue by Region")
build_report(
    orders_df, 
    ["region"], 
    ["total_orders", "total_revenue"]
).show()



Report 1: Revenue by Region
+-------+------------+--------------+
| region|total_orders| total_revenue|
+-------+------------+--------------+
|  South|       99658|1.2171369343E8|
|Central|      100135|1.2212838627E8|
|   East|       99856|1.2165980386E8|
|   West|       99709| 1.215369955E8|
|  North|      100642|1.2326413535E8|
+-------+------------+--------------+



In [ ]:
# User request 2: Customer metrics by channel
print("Report 2: Customer Metrics by Channel")
build_report(
    orders_df,
    ["channel"],
    ["unique_customers", "avg_order_value", "total_orders"]
).orderBy(F.desc("unique_customers")).show()

Report 2: Customer Metrics by Channel
+----------+----------------+---------------+------------+
|   channel|unique_customers|avg_order_value|total_orders|
+----------+----------------+---------------+------------+
|Mobile App|           45967|        1219.34|      124897|
|    Online|           45938|        1217.46|      125056|
|     Phone|           45920|        1224.92|      125277|
|     Store|           45872|         1220.7|      124770|
+----------+----------------+---------------+------------+



In [ ]:
# User request 2: Customer metrics by channel
print("Report 2: Customer Metrics by Channel and Region")
build_report(
    orders_df,
    ["channel", "region"],
    ["unique_customers", "avg_order_value", "total_orders"]
).orderBy(F.desc("unique_customers")).show()

Report 2: Customer Metrics by Channel
+----------+-------+----------------+---------------+------------+
|   channel| region|unique_customers|avg_order_value|total_orders|
+----------+-------+----------------+---------------+------------+
|     Store|  North|           19892|        1226.63|       25365|
|     Phone|Central|           19876|        1220.92|       25224|
|Mobile App|  North|           19868|        1225.71|       25275|
|    Online|Central|           19853|        1220.48|       25316|
|     Store|   West|           19745|         1219.5|       24922|
|    Online|  North|           19740|        1220.02|       25014|
|     Store|   East|           19732|        1216.27|       25055|
|     Phone|  North|           19708|        1226.72|       24988|
|    Online|  South|           19696|        1216.01|       24971|
|     Phone|   West|           19692|         1222.6|       25064|
|    Online|   East|           19690|        1213.28|       24927|
|     Phone|   East|    

### Reusable Transformations

One of the most powerful features of the DataFrame API is the ability to create reusable transformation functions. These can be tested, documented, and shared across projects.


In [ ]:
# Cell 27: Simple Reusable Transformation

# A function that adds a computed column


+----------+--------+----------+------------------+
|  order_id|quantity|unit_price|        line_total|
+----------+--------+----------+------------------+
|ORD0000000|      12|     86.13|           1033.56|
|ORD0000001|      12|    268.19|3218.2799999999997|
|ORD0000002|       2|      69.5|             139.0|
|ORD0000003|      12|    165.76|           1989.12|
|ORD0000004|      13|    242.72|           3155.36|
+----------+--------+----------+------------------+
only showing top 5 rows



In [70]:
##Execute cuntions on dataframes 2:
add_line_total(orders_df).show(5)

+----------+-----------+----------+--------+----------+----------+----------+-------+------------------+
|  order_id|customer_id|product_id|quantity|unit_price|order_date|   channel| region|        line_total|
+----------+-----------+----------+--------+----------+----------+----------+-------+------------------+
|ORD0000000|  CUST07297|   PROD007|      12|     86.13|2024-03-22|     Store|  South|           1033.56|
|ORD0000001|  CUST06718|   PROD174|      12|    268.19|2024-04-04|    Online|Central|3218.2799999999997|
|ORD0000002|  CUST02083|   PROD008|       2|      69.5|2024-02-24|    Online|Central|             139.0|
|ORD0000003|  CUST46926|   PROD167|      12|    165.76|2024-01-26|     Store|   West|           1989.12|
|ORD0000004|  CUST18232|   PROD002|      13|    242.72|2024-03-16|Mobile App|   East|           3155.36|
+----------+-----------+----------+--------+----------+----------+----------+-------+------------------+
only showing top 5 rows



In [ ]:
# Transformation with Parameters

# Sometimes transformations need parameters
def categorise_value(df, value_column, new_column_name, thresholds):
    """
    Categorise numeric values into buckets.
    
    Args:
        df: Input DataFrame
        value_column: Column to categorise
        new_column_name: Name for the new category column
        thresholds: Dict with threshold values, e.g., {"Low": 0, "Medium": 100, "High": 500}
    """
    # Sort thresholds by value (descending) to build WHEN clauses correctly
    sorted_thresholds = sorted(thresholds.items(), key=lambda x: x[1], reverse=True)
    
    # Build the WHEN expression
    condition = None
    for label, threshold in sorted_thresholds:
        if condition is None:
            condition = F.when(F.col(value_column) >= threshold, label)
        else:
            condition = condition.when(F.col(value_column) >= threshold, label)
        # PRINT THE LOGIC: This shows the trainees the CASE WHEN structure
        print(f"Iteration ({label}):")
        print(condition) 
        print("-" * 20)
    return df.withColumn(new_column_name, condition)



In [74]:
add_line_total(orders_df).show(5)

+----------+-----------+----------+--------+----------+----------+----------+-------+------------------+
|  order_id|customer_id|product_id|quantity|unit_price|order_date|   channel| region|        line_total|
+----------+-----------+----------+--------+----------+----------+----------+-------+------------------+
|ORD0000000|  CUST07297|   PROD007|      12|     86.13|2024-03-22|     Store|  South|           1033.56|
|ORD0000001|  CUST06718|   PROD174|      12|    268.19|2024-04-04|    Online|Central|3218.2799999999997|
|ORD0000002|  CUST02083|   PROD008|       2|      69.5|2024-02-24|    Online|Central|             139.0|
|ORD0000003|  CUST46926|   PROD167|      12|    165.76|2024-01-26|     Store|   West|           1989.12|
|ORD0000004|  CUST18232|   PROD002|      13|    242.72|2024-03-16|Mobile App|   East|           3155.36|
+----------+-----------+----------+--------+----------+----------+----------+-------+------------------+
only showing top 5 rows



In [73]:
categorise_value(orders_with_total, 
            "line_total", 
            "order_size",
            {"Small": 0, "Medium": 100, "Large": 500, "Premium": 1000})

Iteration (Premium):
Column<'CASE WHEN (line_total >= 1000) THEN Premium END'>
--------------------
Iteration (Large):
Column<'CASE WHEN (line_total >= 1000) THEN Premium WHEN (line_total >= 500) THEN Large END'>
--------------------
Iteration (Medium):
Column<'CASE WHEN (line_total >= 1000) THEN Premium WHEN (line_total >= 500) THEN Large WHEN (line_total >= 100) THEN Medium END'>
--------------------
Iteration (Small):
Column<'CASE WHEN (line_total >= 1000) THEN Premium WHEN (line_total >= 500) THEN Large WHEN (line_total >= 100) THEN Medium WHEN (line_total >= 0) THEN Small END'>
--------------------


DataFrame[order_id: string, customer_id: string, product_id: string, quantity: bigint, unit_price: double, order_date: string, channel: string, region: string, line_total: double, order_size: string]

In [75]:
categorise_value( 
                    add_line_total(orders_df),
                    "line_total", 
                    "order_size",
                    {"Small": 0, "Medium": 100, "Large": 500, "Premium": 1000}
).show(5)

Iteration (Premium):
Column<'CASE WHEN (line_total >= 1000) THEN Premium END'>
--------------------
Iteration (Large):
Column<'CASE WHEN (line_total >= 1000) THEN Premium WHEN (line_total >= 500) THEN Large END'>
--------------------
Iteration (Medium):
Column<'CASE WHEN (line_total >= 1000) THEN Premium WHEN (line_total >= 500) THEN Large WHEN (line_total >= 100) THEN Medium END'>
--------------------
Iteration (Small):
Column<'CASE WHEN (line_total >= 1000) THEN Premium WHEN (line_total >= 500) THEN Large WHEN (line_total >= 100) THEN Medium WHEN (line_total >= 0) THEN Small END'>
--------------------
+----------+-----------+----------+--------+----------+----------+----------+-------+------------------+----------+
|  order_id|customer_id|product_id|quantity|unit_price|order_date|   channel| region|        line_total|order_size|
+----------+-----------+----------+--------+----------+----------+----------+-------+------------------+----------+
|ORD0000000|  CUST07297|   PROD007|      

In [ ]:
# Cell 28: Transformation with Parameters

# Sometimes transformations need parameters


+----------+------+
|order_size| count|
+----------+------+
|     Small| 29067|
|     Large|104608|
|    Medium|127327|
|   Premium|238998|
+----------+------+



In [81]:
#Date-Based Transformations

def add_date_parts(df, date_column):
    """Add useful date components from a date column"""
    return (
        df
        .withColumn("year", F.year(date_column))
        .withColumn("month", F.month(date_column))
        .withColumn("day_of_week", F.dayofweek(date_column))
        .withColumn("week_of_year", F.weekofyear(date_column))
        .withColumn("is_weekend", F.dayofweek(date_column).isin([1, 7]))
    )

def add_fiscal_columns(df, date_column, fiscal_year_start_month=4):
    """
    Add fiscal year and quarter based on configurable fiscal year start.
    
    Args:
        df: Input DataFrame
        date_column: Name of the date column
        fiscal_year_start_month: Month when fiscal year starts (default: April = 4)
    """
    return (
        df
        .withColumn(
            "fiscal_year",
            F.when(
                F.month(date_column) >= fiscal_year_start_month,
                F.year(date_column)
            ).otherwise(F.year(date_column) - 1)
        )
        .withColumn(
            "fiscal_quarter",
            F.ceil(
                ((F.month(date_column) - fiscal_year_start_month + 12) % 12 + 1) / 3
            ).cast("int")
        )
    )


In [82]:
enriched = add_fiscal_columns(
                    add_date_parts(
                    add_line_total(orders_df),
                    "order_date"),
                    "order_date"
                )


In [ ]:
enriched.show(4)

+----------+-----------+----------+--------+----------+----------+-------+-------+------------------+----+-----+-----------+------------+----------+-----------+--------------+
|  order_id|customer_id|product_id|quantity|unit_price|order_date|channel| region|        line_total|year|month|day_of_week|week_of_year|is_weekend|fiscal_year|fiscal_quarter|
+----------+-----------+----------+--------+----------+----------+-------+-------+------------------+----+-----+-----------+------------+----------+-----------+--------------+
|ORD0000000|  CUST07297|   PROD007|      12|     86.13|2024-03-22|  Store|  South|           1033.56|2024|    3|          6|          12|     false|       2023|             4|
|ORD0000001|  CUST06718|   PROD174|      12|    268.19|2024-04-04| Online|Central|3218.2799999999997|2024|    4|          5|          14|     false|       2024|             1|
|ORD0000002|  CUST02083|   PROD008|       2|      69.5|2024-02-24| Online|Central|             139.0|2024|    2|        

In [ ]:
#Data Quality Transformations
"""Trim whitespace and convert to uppercase for specified columns"""



In [87]:
standardise_text_columns(enriched, ['channel', 'region']).show(3)

+----------+-----------+----------+--------+----------+----------+-------+-------+------------------+----+-----+-----------+------------+----------+-----------+--------------+
|  order_id|customer_id|product_id|quantity|unit_price|order_date|channel| region|        line_total|year|month|day_of_week|week_of_year|is_weekend|fiscal_year|fiscal_quarter|
+----------+-----------+----------+--------+----------+----------+-------+-------+------------------+----+-----+-----------+------------+----------+-----------+--------------+
|ORD0000000|  CUST07297|   PROD007|      12|     86.13|2024-03-22|  STORE|  SOUTH|           1033.56|2024|    3|          6|          12|     false|       2023|             4|
|ORD0000001|  CUST06718|   PROD174|      12|    268.19|2024-04-04| ONLINE|CENTRAL|3218.2799999999997|2024|    4|          5|          14|     false|       2024|             1|
|ORD0000002|  CUST02083|   PROD008|       2|      69.5|2024-02-24| ONLINE|CENTRAL|             139.0|2024|    2|        

## Hybrid Patterns: Best of Both Worlds

### Introduction

In real-world data engineering, you'll rarely use just SQL or just DataFrame API exclusively. The most effective pipelines leverage both, using each where it shines:

- **DataFrame API** for data preparation, reusable transformations, and programmatic control
- **SQL** for complex analytics, reporting queries, and operations familiar to the broader team

This section presents common hybrid patterns you'll encounter and should adopt.

### Pattern 1: DataFrame for Preparation, SQL for Analysis

This is the most common hybrid pattern. Use DataFrame API to prepare and enrich your data, then register it as a view for SQL-based analysis.


In [ ]:
# Complex example

# DataFrame preparation pipeline
prepared_data = (
    orders_df
    # Add computed columns
    .withColumn("line_total", F.col("quantity") * F.col("unit_price"))
    .withColumn("order_month", F.date_format("order_date", "yyyy-MM"))
    .withColumn("is_weekend", F.dayofweek("order_date").isin([1, 7]))
    # Join with products for category
    .join(products_df.select("product_id", "category"), "product_id")
    # Join with customers for membership
    .join(customers_df.select("customer_id", "membership_level"), "customer_id")
    # Add order size tier
    .withColumn(
        "order_tier",
        F.when(F.col("line_total") >= 500, "Premium")
        .when(F.col("line_total") >= 100, "Standard")
        .otherwise("Budget")
    )
)

In [62]:
# Register the prepared data
prepared_data.createOrReplaceTempView("prepared_orders")

In [63]:
# Now analysts can write clear SQL against enriched data
analysis = spark.sql("""
    SELECT 
        order_month,
        category,
        order_tier,
        membership_level,
        COUNT(*) AS order_count,
        ROUND(SUM(line_total), 2) AS total_revenue,
        ROUND(AVG(line_total), 2) AS avg_order_value,
        SUM(CASE WHEN is_weekend THEN 1 ELSE 0 END) AS weekend_orders
    FROM prepared_orders
    GROUP BY order_month, category, order_tier, membership_level
    ORDER BY order_month, category, order_tier
""")

print("Analysis on Prepared Data:")
analysis.show(20)

Analysis on Prepared Data:
+-----------+--------+----------+----------------+-----------+-------------+---------------+--------------+
|order_month|category|order_tier|membership_level|order_count|total_revenue|avg_order_value|weekend_orders|
+-----------+--------+----------+----------------+-----------+-------------+---------------+--------------+
|    2024-01|   Books|    Budget|          Silver|        337|     19956.27|          59.22|            83|
|    2024-01|   Books|    Budget|        Platinum|        336|     19483.39|          57.99|            91|
|    2024-01|   Books|    Budget|          Bronze|        316|     18447.01|          58.38|            88|
|    2024-01|   Books|    Budget|            Gold|        296|     18135.77|          61.27|            82|
|    2024-01|   Books|   Premium|            Gold|       3842|    6466681.9|        1683.16|           986|
|    2024-01|   Books|   Premium|          Silver|       3798|   6384293.93|        1680.96|           985|
|

### Pattern 2: SQL Subqueries as DataFrames

Sometimes you want a complex SQL query as a starting point, then continue with DataFrame operations.

In [ ]:
# SQL for the complex aggregation
customer_summary_sql = spark.sql("""
    SELECT 
        customer_id,
        COUNT(*) AS order_count,
        SUM(quantity * unit_price) AS total_spend
    FROM orders
    GROUP BY customer_id
""")

# Continue with DataFrame for additional processing
customer_segments = (
    customer_summary_sql
    .withColumn(
        "segment",
        F.when(F.col("total_spend") >= 5000, "VIP")
        .when(F.col("total_spend") >= 1000, "Regular")
        .otherwise("Occasional")
    )
)

customer_segments.groupBy("segment").count().show()

### Pattern 3: Dynamic SQL Generation

When you need SQL's readability but with dynamic elements, generate SQL strings from Python.

In [ ]:
# Dynamically select which columns to aggregate
metrics_needed = ["SUM", "AVG", "COUNT"]

# Build the SELECT clause
select_parts = ["region"]
for metric in metrics_needed:
    if metric == "SUM":
        select_parts.append("ROUND(SUM(quantity * unit_price), 2) AS total_revenue")
    elif metric == "AVG":
        select_parts.append("ROUND(AVG(quantity * unit_price), 2) AS avg_order_value")
    elif metric == "COUNT":
        select_parts.append("COUNT(*) AS order_count")

query = f"""
    SELECT {', '.join(select_parts)}
    FROM orders
    GROUP BY region
    ORDER BY region
    """

print("Generated SQL:")
print(query)

Generated SQL:

    SELECT region, ROUND(SUM(quantity * unit_price), 2) AS total_revenue, ROUND(AVG(quantity * unit_price), 2) AS avg_order_value, COUNT(*) AS order_count
    FROM orders
    GROUP BY region
    ORDER BY region
    

Result:
+-------+--------------+---------------+-----------+
| region| total_revenue|avg_order_value|order_count|
+-------+--------------+---------------+-----------+
|Central|1.2212838627E8|        1219.64|     100135|
|   East|1.2165980386E8|        1218.35|      99856|
|  North|1.2326413535E8|        1224.78|     100642|
|  South|1.2171369343E8|        1221.31|      99658|
|   West| 1.215369955E8|        1218.92|      99709|
+-------+--------------+---------------+-----------+



In [ ]:
spark.sql(query).show()

In [ ]:
# Report Builder

def generate_report_sql(group_by, metrics, filters=None, order_by=None, limit=None):
    """
    Generate a SQL report query from parameters.
    
    Args:
        group_by: List of columns to group by
        metrics: List of metric names to include
        filters: Dict of {column: value} filters
        order_by: Column to order by (prefix with - for DESC)
        limit: Max rows to return
    """
    # Metric definitions
    metric_sql = {
        "revenue": "ROUND(SUM(quantity * unit_price), 2) AS revenue",
        "orders": "COUNT(*) AS orders",
        "avg_value": "ROUND(AVG(quantity * unit_price), 2) AS avg_value",
        "unique_customers": "COUNT(DISTINCT customer_id) AS unique_customers",
        "avg_quantity": "ROUND(AVG(quantity), 1) AS avg_quantity"
    }
    
    # Build SELECT
    select_cols = group_by + [metric_sql[m] for m in metrics if m in metric_sql]
    select_clause = ", ".join(select_cols)
    # Build WHERE
    where_clause = ""
    if filters:
        conditions = [f"{col} = '{val}'" for col, val in filters.items()]
        where_clause = "WHERE " + " AND ".join(conditions)
    # Build GROUP BY
    group_clause = "GROUP BY " + ", ".join(group_by)
    # Build ORDER BY
    order_clause = ""
    if order_by:
        if order_by.startswith("-"):
            order_clause = f"ORDER BY {order_by[1:]} DESC"
        else:
            order_clause = f"ORDER BY {order_by}"
    # Build LIMIT
    limit_clause = f"LIMIT {limit}" if limit else ""
    return f"""
        SELECT {select_clause}
        FROM orders
        {where_clause}
        {group_clause}
        {order_clause}
        {limit_clause}
    """


In [96]:

# Generate and run reports
print("=== Report 1: Revenue by Region ===")
sql1 = generate_report_sql(
    group_by=["region"],
    metrics=["revenue", "orders"],
    order_by="revenue"
)
spark.sql(sql1).show()

=== Report 1: Revenue by Region ===

+-------+--------------+------+
| region|       revenue|orders|
+-------+--------------+------+
|   West| 1.215369955E8| 99709|
|   East|1.2165980386E8| 99856|
|  South|1.2171369343E8| 99658|
|Central|1.2212838627E8|100135|
|  North|1.2326413535E8|100642|
+-------+--------------+------+



In [97]:
print("=== Report 2: Channel Performance (Online only) ===")
sql2 = generate_report_sql(
    group_by=["channel", "region"],
    metrics=["revenue", "unique_customers", "avg_value"],
    filters={"channel": "Online"},
    order_by="revenue",
    limit=10
)
spark.sql(sql2).show()

=== Report 2: Channel Performance (Online only) ===
LIMIT 10
+-------+-------+-------------+----------------+---------+
|channel| region|      revenue|unique_customers|avg_value|
+-------+-------+-------------+----------------+---------+
| Online|   West|3.022677358E7|           19568|  1217.45|
| Online|   East|3.024342241E7|           19690|  1213.28|
| Online|  South|3.036502296E7|           19696|  1216.01|
| Online|  North|3.051762028E7|           19740|  1220.02|
| Online|Central|3.089767387E7|           19853|  1220.48|
+-------+-------+-------------+----------------+---------+



```
┌─────────────────────────────────────────────────────────────────────┐
│                    WHICH APPROACH SHOULD I USE?                      │
├─────────────────────────────────────────────────────────────────────┤
│                                                                      │
│  Use SQL when:                                                       │
│  ├── Writing complex analytical queries (window functions, CTEs)    │
│  ├── The query will be reviewed by SQL-familiar analysts            │
│  ├── Porting existing SQL from a data warehouse                     │
│  ├── Expressing data validation rules                               │
│  └── The query is relatively static (not parameterised heavily)     │
│                                                                      │
│  Use DataFrame API when:                                             │
│  ├── Building reusable transformation functions                     │
│  ├── Dynamic column selection or transformations                    │
│  ├── Integrating with Python libraries (pandas UDFs, ML)            │
│  ├── Need IDE support and type checking                             │
│  └── Building data pipelines with unit tests                        │
│                                                                      │
│  Use Hybrid when:                                                    │
│  ├── Complex pipeline with both ETL and analytics                   │
│  ├── Team has mixed SQL/Python expertise                            │
│  ├── Want to leverage strengths of both approaches                  │
│  └── Clear separation between data prep and analysis                │
│                                                                      │
└─────────────────────────────────────────────────────────────────────┘
```

## Exercises

### Exercise 1: Refactoring SQL to DataFrame API

The analytics team wrote this SQL query. The data engineering team needs to convert it to DataFrame API for integration into the automated pipeline with unit tests.

In [9]:
add_line_total(orders_df).createOrReplaceTempView("prepared_orders")


In [10]:

original_sql = """
SELECT 
    region,
    channel,
    COUNT(order_id) AS total_orders,
    -- Calculate revenue only for orders > 500
    ROUND(SUM(CASE WHEN line_total > 500 THEN line_total ELSE 0 END), 2) AS premium_revenue,
    -- Calculate what % of orders were 'Online'
    ROUND(AVG(CASE WHEN channel = 'Online' THEN 1 ELSE 0 END) * 100, 1) AS online_pct,
    ROUND(SUM(line_total), 2) AS total_revenue
FROM prepared_orders
GROUP BY region, channel
HAVING total_orders > 5
ORDER BY total_revenue DESC
"""

spark.sql(original_sql).show()

+-------+----------+------------+---------------+----------+-------------+
| region|   channel|total_orders|premium_revenue|online_pct|total_revenue|
+-------+----------+------------+---------------+----------+-------------+
|  North|     Store|       25365|  2.918193217E7|       0.0|3.111340415E7|
|  North|Mobile App|       25275|  2.908437644E7|       0.0|3.097992653E7|
|Central|    Online|       25316|  2.897665186E7|     100.0|3.089767387E7|
|Central|     Phone|       25224|  2.888817793E7|       0.0|3.079650076E7|
|   East|     Phone|       25023|  2.883275507E7|       0.0|3.068584427E7|
|  South|     Phone|       24978|  2.878659377E7|       0.0|3.067550189E7|
|  North|     Phone|       24988|  2.878443833E7|       0.0|3.065318439E7|
|   West|     Phone|       25064|  2.875697581E7|       0.0| 3.06431578E7|
|  North|    Online|       25014|  2.863379712E7|     100.0|3.051762028E7|
|   East|     Store|       25055|  2.857585903E7|       0.0|3.047355793E7|
|  South|Mobile App|     

In [ ]:
# Cell 23: Exercise 1 - YOUR SOLUTION


In [ ]:
# --- Testing ---
# Scenario A: Standard Report
print("Standard Report (>$500):")
generate_performance_report(add_line_total(orders_df)).show(5)


Standard Report (>$500):
+-------+----------+------------+---------------+----------+-------------+
| region|   channel|total_orders|premium_revenue|online_pct|total_revenue|
+-------+----------+------------+---------------+----------+-------------+
|  North|     Store|       25365|    29181932.17|       0.0|  31113404.15|
|  North|Mobile App|       25275|    29084376.44|       0.0|  30979926.53|
|Central|    Online|       25316|    28976651.86|     100.0|  30897673.87|
|Central|     Phone|       25224|    28888177.93|       0.0|  30796500.76|
|   East|     Phone|       25023|    28832755.07|       0.0|  30685844.27|
+-------+----------+------------+---------------+----------+-------------+
only showing top 5 rows



In [ ]:

# Scenario B
print("Small-Ticket High-Volume Report (>$100):")
generate_performance_report(add_line_total(orders_df), premium_threshold=100, min_orders=20).show(5)

Small-Ticket High-Volume Report (>$100):
+-------+----------+------------+---------------+----------+-------------+
| region|   channel|total_orders|premium_revenue|online_pct|total_revenue|
+-------+----------+------------+---------------+----------+-------------+
|  North|     Store|       25365|    31027496.83|       0.0|  31113404.15|
|  North|Mobile App|       25275|    30894748.69|       0.0|  30979926.53|
|Central|    Online|       25316|    30812064.94|     100.0|  30897673.87|
|Central|     Phone|       25224|    30708491.89|       0.0|  30796500.76|
|   East|     Phone|       25023|    30599873.70|       0.0|  30685844.27|
+-------+----------+------------+---------------+----------+-------------+
only showing top 5 rows




### Exercise 2: The Global Currency Converter

The analytics team wrote this SQL query. The data engineering team needs to convert it to DataFrame API for integration into the automated pipeline with unit tests.

**Goal:** Build a robust currency conversion engine that evolves from a static multiplier to a dynamic, time-aware lookup system.

- Phase 1: The Simple Multiplier: Task: Create a basic Python function that converts USD to EUR using a fixed rate (e.g., 0.92) and apply it to the orders data.


- Phase 2: The Multi-Currency Dictionary: Task: Update the function to handle multiple currencies. Instead of a hard-coded number, use a dictionary.

- Phase 3: The Time-Aware Converter (Final Challenge): 
    - Scenario: Exchange rates fluctuate every month. We need to convert prices based on the Year and Month the order was placed. 
    - Question: Which product categories are most profitable across different customer membership tiers when adjusted for local currency rates?


In [ ]:
##SOLUTION Phase 1:
##Your code: 


# Usage:
print("Phase 1: Simple DataFrame Transformation")
phase1_df = convert_to_eur_simple(orders_df, "unit_price", "price_eur")
phase1_df.select("order_id", "unit_price", "price_eur").show(5)

Phase 1: Simple DataFrame Transformation
+----------+----------+---------+
|  order_id|unit_price|price_eur|
+----------+----------+---------+
|ORD0000000|     86.13|    79.24|
|ORD0000001|    268.19|   246.73|
|ORD0000002|      69.5|    63.94|
|ORD0000003|    165.76|    152.5|
|ORD0000004|    242.72|    223.3|
+----------+----------+---------+
only showing top 5 rows



In [ ]:
##SOLUTION Phase 2:
RATES = {"EUR": 0.92, "GBP": 0.78, "JPY": 150.10}

##Your code: 


# Usage:
print("Phase 2: Dictionary-based Conversion")
phase2_df = convert_by_currency(orders_df, "unit_price", "GBP", "price_gbp")
phase2_df.select("order_id", "unit_price", "price_gbp").show(5)

Phase 2: Dictionary-based Conversion
+----------+----------+---------+
|  order_id|unit_price|price_gbp|
+----------+----------+---------+
|ORD0000000|     86.13|    67.18|
|ORD0000001|    268.19|   209.19|
|ORD0000002|      69.5|    54.21|
|ORD0000003|    165.76|   129.29|
|ORD0000004|    242.72|   189.32|
+----------+----------+---------+
only showing top 5 rows



## Key Takeaways

### Summary

1. **SQL and DataFrame API produce identical execution plans** - Choose based on readability and maintainability, not performance.

2. **Use temporary views as the bridge** - `createOrReplaceTempView()` lets you move between paradigms freely.

3. **Match the approach to the task:**
   - SQL for analytics, window functions, and familiarity
   - DataFrame API for reusability, testing, and dynamic control
   - Hybrid for complex real-world pipelines

4. **Protect against SQL injection** - Use DataFrame filters or validated inputs, never raw string concatenation with user data.

5. **Document your choices** - Future maintainers will thank you for explaining why you chose SQL vs DataFrame for each section.

---


## Further Reading

To deepen your understanding of hybrid SQL and DataFrame patterns in PySpark, consider the following resources:

- **PySpark SQL Functions Reference:** Complete reference of all built-in PySpark SQL functions including string, date, aggregate, and conditional functions. Essential for looking up function syntax and parameters. [https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html)

- **Spark SQL Programming Guide:** Official Apache Spark documentation covering DataFrames, SQL queries, data sources, and the Catalyst optimizer. Provides foundational understanding of how Spark SQL works under the hood. [https://spark.apache.org/docs/latest/sql-programming-guide.html](https://spark.apache.org/docs/latest/sql-programming-guide.html)

- **PySpark DataFrame API:** Comprehensive documentation of all DataFrame methods including transformations, actions, joins, and aggregations with practical code examples. [https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/dataframe.html](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/dataframe.html)

- **Databricks PySpark Basics:** Practical guide from Databricks covering DataFrame creation, transformations, joins, and aggregations with clear examples. Good for reinforcing concepts with hands-on patterns. [https://docs.databricks.com/en/pyspark/basics](https://docs.databricks.com/en/pyspark/basics)


In [ ]:
# Cleanup

# Drop temporary views
views_to_drop = [
    "orders", "products", "customers", 
    "order_totals", "enriched_orders", "orders_prepped",
    "high_value_orders", "prepared_orders", "prepped_orders",
    "transactions", "stores", "store_trans"
]

for view in views_to_drop:
    try:
        spark.catalog.dropTempView(view)
    except:
        pass

# Clear cache
spark.catalog.clearCache()
